In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Day 3 basic libraries imported successfully")

Day 3 basic libraries imported successfully


In [2]:
df = pd.read_csv("../data/frix_features_day1.csv")

print("Dataset loaded successfully")
print(df.shape)
df.head()

Dataset loaded successfully
(1048575, 20)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,origin_balance_error,dest_balance_error,is_high_risk_type,sender_txn_count,receiver_txn_count,sender_emptied_account,is_large_amount,risk_score_v1,rule_alert_v1
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0.0,9839.64,0,1,1,0,0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0.0,1864.28,0,1,1,0,0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0.0,181.00,1,1,26,1,0,45,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,0.0,21363.00,1,1,27,1,0,45,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0.0,11668.14,0,1,1,0,0,0,0


In [3]:
graph_df = df[[
    "nameOrig",
    "nameDest",
    "amount",
    "type",
    "isFraud"
]].copy()

graph_df.head()

,nameOrig,nameDest,amount,type,isFraud
0,C1231006815,M1979787155,9839.64,PAYMENT,0
1,C1666544295,M2044282225,1864.28,PAYMENT,0
2,C1305486145,C553264065,181.00,TRANSFER,1
3,C840083671,C38997010,181.00,CASH_OUT,1
4,C2048537720,M1230701703,11668.14,PAYMENT,0


In [4]:
graph_df = df[[
    "nameOrig",
    "nameDest",
    "amount",
    "type",
    "isFraud"
]].copy()

print("Original df rows:", df.shape[0])
print("Graph df rows:", graph_df.shape[0])
graph_df.head()

Original df rows: 1048575
Graph df rows: 1048575


,nameOrig,nameDest,amount,type,isFraud
0,C1231006815,M1979787155,9839.64,PAYMENT,0
1,C1666544295,M2044282225,1864.28,PAYMENT,0
2,C1305486145,C553264065,181.00,TRANSFER,1
3,C840083671,C38997010,181.00,CASH_OUT,1
4,C2048537720,M1230701703,11668.14,PAYMENT,0


In [5]:
receiver_in_degree = graph_df.groupby("nameDest")["nameOrig"].nunique()
sender_out_degree = graph_df.groupby("nameOrig")["nameDest"].nunique()

print("Receiver in-degree calculated:", receiver_in_degree.shape)
print("Sender out-degree calculated:", sender_out_degree.shape)

Receiver in-degree calculated: (449635,)
Sender out-degree calculated: (1048317,)


In [6]:
graph_df["receiver_in_degree"] = graph_df["nameDest"].map(receiver_in_degree)
graph_df["sender_out_degree"] = graph_df["nameOrig"].map(sender_out_degree)

graph_df[["nameOrig", "nameDest", "receiver_in_degree", "sender_out_degree", "isFraud"]].head()

,nameOrig,nameDest,receiver_in_degree,sender_out_degree,isFraud
0,C1231006815,M1979787155,1,1,0
1,C1666544295,M2044282225,1,1,0
2,C1305486145,C553264065,26,1,1
3,C840083671,C38997010,27,1,1
4,C2048537720,M1230701703,1,1,0


In [7]:
graph_df.groupby("isFraud")[["receiver_in_degree", "sender_out_degree"]].mean()

,receiver_in_degree,sender_out_degree
isFraud,,
0,10.145925,1.000493
1,7.495622,1.000000


In [8]:
receiver_fraud_count = graph_df.groupby("nameDest")["isFraud"].sum()
receiver_total_amount = graph_df.groupby("nameDest")["amount"].sum()
receiver_avg_amount = graph_df.groupby("nameDest")["amount"].mean()

print("Receiver fraud count calculated:", receiver_fraud_count.shape)
print("Receiver total amount calculated:", receiver_total_amount.shape)
print("Receiver average amount calculated:", receiver_avg_amount.shape)

Receiver fraud count calculated: (449635,)
Receiver total amount calculated: (449635,)
Receiver average amount calculated: (449635,)


In [9]:
graph_df["receiver_fraud_count"] = graph_df["nameDest"].map(receiver_fraud_count)
graph_df["receiver_total_amount"] = graph_df["nameDest"].map(receiver_total_amount)
graph_df["receiver_avg_amount"] = graph_df["nameDest"].map(receiver_avg_amount)

graph_df[[
    "nameDest",
    "receiver_in_degree",
    "receiver_fraud_count",
    "receiver_total_amount",
    "receiver_avg_amount",
    "isFraud"
]].head()

,nameDest,receiver_in_degree,receiver_fraud_count,receiver_total_amount,receiver_avg_amount,isFraud
0,M1979787155,1,0,9839.64,9839.640000,0
1,M2044282225,1,0,1864.28,1864.280000,0
2,C553264065,26,1,5653724.70,217450.950000,1
3,C38997010,27,1,7796257.76,288750.287407,1
4,M1230701703,1,0,11668.14,11668.140000,0


In [10]:
receiver_in_degree_threshold = graph_df["receiver_in_degree"].quantile(0.95)
receiver_total_amount_threshold = graph_df["receiver_total_amount"].quantile(0.95)

print("Receiver in-degree 95th percentile:", receiver_in_degree_threshold)
print("Receiver total amount 95th percentile:", receiver_total_amount_threshold)

Receiver in-degree 95th percentile: 31.0
Receiver total amount 95th percentile: 9777055.120000001


In [11]:
graph_df["possible_mule_receiver"] = (
    (graph_df["receiver_in_degree"] > receiver_in_degree_threshold) &
    (graph_df["receiver_total_amount"] > receiver_total_amount_threshold)
).astype(int)

graph_df.groupby("isFraud")["possible_mule_receiver"].mean() * 100

isFraud
0    3.226937
1    2.802102
Name: possible_mule_receiver, dtype: float64

In [12]:
graph_df["is_high_risk_txn_type"] = graph_df["type"].isin(["TRANSFER", "CASH_OUT"]).astype(int)

receiver_high_risk_txn_count = graph_df.groupby("nameDest")["is_high_risk_txn_type"].sum()

graph_df["receiver_high_risk_txn_count"] = graph_df["nameDest"].map(receiver_high_risk_txn_count)

graph_df[[
    "nameDest",
    "type",
    "is_high_risk_txn_type",
    "receiver_high_risk_txn_count",
    "isFraud"
]].head()

,nameDest,type,is_high_risk_txn_type,receiver_high_risk_txn_count,isFraud
0,M1979787155,PAYMENT,0,0,0
1,M2044282225,PAYMENT,0,0,0
2,C553264065,TRANSFER,1,19,1
3,C38997010,CASH_OUT,1,20,1
4,M1230701703,PAYMENT,0,0,0


In [13]:
graph_df.groupby("isFraud")["receiver_high_risk_txn_count"].mean()

isFraud
0    6.618478
1    5.320490
Name: receiver_high_risk_txn_count, dtype: float64

In [14]:
graph_df["receiver_fraud_exposure"] = (graph_df["receiver_fraud_count"] > 0).astype(int)

graph_df.groupby("isFraud")["receiver_fraud_exposure"].mean() * 100

isFraud
0      0.689877
1    100.000000
Name: receiver_fraud_exposure, dtype: float64

In [15]:
graph_df["mule_risk_score_v1"] = (
    (graph_df["receiver_in_degree"] > graph_df["receiver_in_degree"].quantile(0.90)).astype(int) * 25 +
    (graph_df["receiver_total_amount"] > graph_df["receiver_total_amount"].quantile(0.90)).astype(int) * 25 +
    (graph_df["receiver_high_risk_txn_count"] > graph_df["receiver_high_risk_txn_count"].quantile(0.90)).astype(int) * 25 +
    (graph_df["receiver_avg_amount"] > graph_df["receiver_avg_amount"].quantile(0.90)).astype(int) * 25
)

graph_df.groupby("isFraud")["mule_risk_score_v1"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,1047433.0,9.534619,23.742335,0.0,0.0,0.0,0.0,100.0
1,1142.0,16.834501,22.897702,0.0,0.0,0.0,25.0,100.0


In [16]:
graph_df["mule_alert_v1"] = (graph_df["mule_risk_score_v1"] >= 50).astype(int)

pd.crosstab(
    graph_df["isFraud"],
    graph_df["mule_alert_v1"],
    rownames=["Actual Fraud"],
    colnames=["Mule Alert"]
)

Mule Alert,0,1
Actual Fraud,,
0,932591,114842
1,1001,141


In [17]:
mule_true_positive = 141
mule_false_positive = 114842
mule_false_negative = 1001

mule_precision = mule_true_positive / (mule_true_positive + mule_false_positive)
mule_recall = mule_true_positive / (mule_true_positive + mule_false_negative)

print("Mule Alert Precision:", mule_precision)
print("Mule Alert Recall:", mule_recall)

Mule Alert Precision: 0.0012262682309558805
Mule Alert Recall: 0.1234676007005254


In [18]:
graph_features = graph_df[[
    "nameOrig",
    "nameDest",
    "receiver_in_degree",
    "sender_out_degree",
    "receiver_total_amount",
    "receiver_avg_amount",
    "receiver_high_risk_txn_count",
    "possible_mule_receiver",
    "mule_risk_score_v1",
    "mule_alert_v1"
]].copy()

graph_features.head()

,nameOrig,nameDest,receiver_in_degree,sender_out_degree,receiver_total_amount,receiver_avg_amount,receiver_high_risk_txn_count,possible_mule_receiver,mule_risk_score_v1,mule_alert_v1
0,C1231006815,M1979787155,1,1,9839.64,9839.640000,0,0,0,0
1,C1666544295,M2044282225,1,1,1864.28,1864.280000,0,0,0,0
2,C1305486145,C553264065,26,1,5653724.70,217450.950000,19,0,50,1
3,C840083671,C38997010,27,1,7796257.76,288750.287407,20,0,75,1
4,C2048537720,M1230701703,1,1,11668.14,11668.140000,0,0,0,0


In [19]:
graph_features.to_csv("../data/frix_graph_features_day3.csv", index=False)

print("Day 3 graph features saved successfully")

Day 3 graph features saved successfully


In [20]:
import os

os.listdir("../data")

['frix_features_day1.csv',
 'frix_graph_features_day3.csv',
 'paysim_transactions.csv.csv']